# Serve embeddings on demand with SageMaker serverless inference

## What serverless inference is

A real-time SageMaker endpoint runs on instances you pick and pay for by the
hour, whether or not any requests arrive. You choose the instance type and
count, attach an autoscaling policy, and keep at least one instance warm so the
endpoint can answer immediately. That is the right shape for steady, predictable
traffic, and wasteful for everything else.

**Serverless inference** takes the instances out of the picture. You give
SageMaker two numbers (how much memory a copy of the model needs and how many
requests it may handle at once) and it provisions compute on demand, scales it
with traffic, and scales it to **zero** when the endpoint is idle. You pay per
millisecond of request processing and for the data processed, and nothing while
no requests are in flight, so an idle on-demand endpoint is free. (The exception
is provisioned concurrency, described below: reserving warm workers bills around
the clock, so with it set the endpoint keeps costing money even while completely
idle.)

![SageMaker serverless inference workflow](https://docs.aws.amazon.com/images/sagemaker/latest/dg/images/serverless-endpoints-how-it-works.png)

The trade-off is the **cold start**. When a request arrives and no compute is
warm (right after deployment, or after an idle stretch), SageMaker has to start
a worker, pull the container, and load the model before it can answer. That
first request is slow; the ones behind it, while the worker stays warm, are
fast. If you cannot tolerate the occasional slow request, *provisioned
concurrency* keeps a set number of workers warm at all times, at the cost of
paying for them whether or not they are used.

## When it fits, and when it doesn't

Serverless is the cost-effective choice when traffic is **intermittent or
unpredictable** and the workload can absorb an occasional cold start:

- Internal tools, dashboards, and low-QPS APIs that sit idle most of the day.
- Dev, test, and staging endpoints you do not want billed around the clock.
- New models whose traffic you cannot forecast yet.

Reach for a provisioned real-time endpoint instead when traffic is **steady and
high** (a busy always-on instance is cheaper per request and has no cold starts),
or when a strict latency SLA leaves no room for one. And mind the hard limits:
serverless is **CPU-only** (no GPUs), memory is capped at **6 GB**, and a single
endpoint handles at most **200 concurrent requests**. Large or GPU-bound models
do not belong here, and long-running or large-payload jobs belong on
[asynchronous inference](https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference.html)
instead.

## The use case we picked

This tutorial serves an **embedding model for a low-traffic semantic-search
box**. Embedding a user's live query is small, synchronous, and latency-shaped,
but on an internal search tool the queries arrive in bursts with long idle gaps
between them, so paying for an always-on instance is hard to justify. That is
exactly the shape serverless was built for. (The offline other half, embedding
a whole corpus in one large batch, is a job for
[asynchronous inference](https://docs.aws.amazon.com/sagemaker/latest/dg/async-inference.html);
this notebook is only the online query side.)

The model is `BAAI/bge-small-en-v1.5`: 384-dimensional vectors and a ~130 MB
download that loads comfortably inside the smallest serverless memory tier. It
is served with
[Text Embeddings Inference (TEI)](https://huggingface.co/docs/text-embeddings-inference),
Hugging Face's container for embedding models.

References:

- [SageMaker serverless inference](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-endpoints.html)
- [Serverless endpoint operations](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-endpoints-create-invoke-update-delete.html)
- [Minimizing cold starts with provisioned concurrency](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-endpoints-autoscale.html)


## Prerequisites

Run the next cell before importing the SDK. It installs the SageMaker Python
SDK into the active kernel.

You also need an existing SageMaker execution role with access to SageMaker,
S3, CloudWatch, and the ECR repository that hosts the TEI container. Serverless
inference is available in a
[subset of AWS regions](https://docs.aws.amazon.com/sagemaker/latest/dg/serverless-endpoints.html).
Run this in one of them.


In [ ]:
%pip install "sagemaker>=3.0.0" matplotlib --upgrade --quiet

In [ ]:
import datetime as dt
import json
import math
import os
import time

import boto3
from botocore.exceptions import ClientError

from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.inference_config import ServerlessInferenceConfig
from sagemaker.serve import ModelBuilder, ModelServer
from sagemaker.serve.builder.schema_builder import SchemaBuilder

## The model and how serverless is sized

A serverless endpoint is configured by two numbers instead of an instance type:

- **Memory** (`MEMORY_SIZE_IN_MB`) must be at least as large as one copy of the
  model plus its container. It has to be one of 1024, 2048, 3072, 4096, 5120, or
  6144 MB, and SageMaker gives the container more vCPUs as you go up.
  `BAAI/bge-small-en-v1.5` is tiny, so 2 GB is ample; a larger embedding model
  would need a larger tier.
- **Max concurrency** (`MAX_CONCURRENCY`) caps how many requests the endpoint
  processes at once, up to 200. Invocations beyond it are throttled rather than
  queued, which keeps one endpoint from consuming your whole account quota.

Leaving `PROVISIONED_CONCURRENCY` unset keeps the endpoint fully on-demand: it
scales to zero when idle and you accept cold starts. Set it to a small integer
(no greater than `MAX_CONCURRENCY`) to keep that many workers permanently warm:
no cold starts, but you pay for the reserved capacity around the clock.

To serve a different model, set `HF_MODEL_ID`, set `EMBEDDING_DIM` to its output
dimension, and raise `MEMORY_SIZE_IN_MB` if the model is larger.


In [ ]:
PROJECT = "hf-serverless-embed"
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%d%H%M%S")

MODEL_ID = os.getenv("HF_MODEL_ID", "BAAI/bge-small-en-v1.5")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "384"))
TEI_VERSION = os.getenv("TEI_VERSION", "1.8.2")
ENDPOINT_NAME = os.getenv("SAGEMAKER_ENDPOINT_NAME", f"{PROJECT}-{RUN_ID}")

# Serverless sizing. Memory must be one of 1024/2048/3072/4096/5120/6144 MB and
# at least the size of one model copy; max concurrency is capped at 200.
MEMORY_SIZE_IN_MB = int(os.getenv("MEMORY_SIZE_IN_MB", "2048"))
MAX_CONCURRENCY = int(os.getenv("MAX_CONCURRENCY", "5"))

# Leave unset for a pure scale-to-zero endpoint; set it to keep workers warm.
_provisioned = os.getenv("PROVISIONED_CONCURRENCY")
PROVISIONED_CONCURRENCY = int(_provisioned) if _provisioned and int(_provisioned) > 0 else None

# Set CLEANUP=false to inspect the endpoint after the tutorial finishes.
CLEANUP = os.getenv("CLEANUP", "true").lower() not in {"0", "false", "no"}

assert MEMORY_SIZE_IN_MB in {1024, 2048, 3072, 4096, 5120, 6144}, MEMORY_SIZE_IN_MB
assert 1 <= MAX_CONCURRENCY <= 200, MAX_CONCURRENCY
assert PROVISIONED_CONCURRENCY is None or PROVISIONED_CONCURRENCY <= MAX_CONCURRENCY

## Set up the SageMaker session

The endpoint runs under a SageMaker execution role: an IAM role that grants
access to S3, ECR, and CloudWatch. Set `SAGEMAKER_EXECUTION_ROLE_ARN` to the
role you want to use, or `SAGEMAKER_EXECUTION_ROLE_NAME` if you only have its
name. Inside SageMaker Studio or a notebook instance you can leave both unset
and the role is detected automatically.


In [ ]:
requested_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION")
boto_session = boto3.Session(region_name=requested_region) if requested_region else boto3.Session()
sess = Session(boto_session=boto_session)
region = sess.boto_region_name

sm = boto_session.client("sagemaker")


def resolve_role(session, sagemaker_session):
    role_arn = os.getenv("SAGEMAKER_EXECUTION_ROLE_ARN")
    if role_arn:
        return role_arn

    role_name = os.getenv("SAGEMAKER_EXECUTION_ROLE_NAME")
    if role_name:
        iam = session.client("iam")
        return iam.get_role(RoleName=role_name)["Role"]["Arn"]

    return get_execution_role(sagemaker_session=sagemaker_session)


role = resolve_role(boto_session, sess)

print(f"region: {region}")
print(f"role: {role}")
print(f"endpoint: {ENDPOINT_NAME}")

## Select the TEI serving container

Serverless inference is CPU-only, so we always use the CPU build of Text
Embeddings Inference, `huggingface-tei-cpu`. `image_uris.retrieve` resolves the
image URI for the current region. No instance type is involved. Serverless has
no instances to size.


In [ ]:
image_uri = image_uris.retrieve(
    framework="huggingface-tei-cpu",
    region=region,
    version=TEI_VERSION,
    image_scope="inference",
)
print(image_uri)

## Build the model

`ModelBuilder` describes the SageMaker model: the Hub model ID, the TEI
container, the model server, and a small input/output example. The container
downloads the model from the Hub when the worker starts. We do not pass an
instance type (serverless provisions its own compute), and the explicit
`image_uri` above is what pins the container. For gated or private models, set
`HF_TOKEN` (or `HUGGING_FACE_HUB_TOKEN`) before running the notebook.


In [ ]:
hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")

env_vars = {
    "HF_MODEL_ID": MODEL_ID,
    "MAX_BATCH_TOKENS": os.getenv("MAX_BATCH_TOKENS", "16384"),
    "MAX_CLIENT_BATCH_SIZE": os.getenv("MAX_CLIENT_BATCH_SIZE", "32"),
}

if hf_token:
    env_vars["HF_TOKEN"] = hf_token
    env_vars["HUGGING_FACE_HUB_TOKEN"] = hf_token

resource_tags = [
    {"Key": "Project", "Value": PROJECT},
    {"Key": "ModelId", "Value": MODEL_ID},
    {"Key": "CreatedBy", "Value": "hf-sagemaker-docs"},
]

model_builder = ModelBuilder(
    model=MODEL_ID,
    role_arn=role,
    sagemaker_session=sess,
    image_uri=image_uri,
    model_server=ModelServer.TEI,
    env_vars=env_vars,
    schema_builder=SchemaBuilder(
        sample_input={"inputs": ["who wrote the origin of species"]},
        sample_output=[[0.0] * EMBEDDING_DIM],
    ),
)

tei_model = model_builder.build(model_name=f"{PROJECT}-model-{RUN_ID}")

## Deploy the serverless endpoint

`ServerlessInferenceConfig` carries the two sizing numbers and, optionally,
provisioned concurrency. Passing it to `deploy` as the `inference_config` is
what makes the endpoint serverless; there is no `instance_type` or
`initial_instance_count` to set. Deployment still takes a few minutes while
SageMaker registers the endpoint; the first *invocation* is where the cold start
shows up.


In [ ]:
serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=MEMORY_SIZE_IN_MB,
    max_concurrency=MAX_CONCURRENCY,
    provisioned_concurrency=PROVISIONED_CONCURRENCY,
)

endpoint = model_builder.deploy(
    endpoint_name=ENDPOINT_NAME,
    inference_config=serverless_config,
    container_timeout_in_seconds=900,
    tags=resource_tags,
    wait=True,
)

endpoint_description = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
endpoint_config_name = endpoint_description["EndpointConfigName"]
model_name = tei_model.model_name

print(f"endpoint status: {endpoint_description['EndpointStatus']}")
print(f"endpoint config: {endpoint_config_name}")
print(f"model: {model_name}")

## Invoke the endpoint and watch the cold start

TEI takes `{"inputs": [...]}` and returns one vector per input. The first call
below lands on a cold endpoint: SageMaker starts a worker and loads the model,
so it is slow. The calls after it reuse the warm worker and return in a fraction
of the time. The gap between them *is* the cold start: the cost you accept in
exchange for scaling to zero.

The numbers vary from run to run, and CloudWatch reports the cold-start portion
separately as the `OverheadLatency` metric. To force another cold start, leave
the endpoint idle for a while and invoke again.


In [ ]:
def embed(texts):
    response = endpoint.invoke(
        body=json.dumps({"inputs": texts}),
        content_type="application/json",
        accept="application/json",
    )
    return json.loads(response.body.read())


def timed_embed(texts):
    start = time.perf_counter()
    vectors = embed(texts)
    return vectors, time.perf_counter() - start


cold_vectors, cold_latency = timed_embed(["who wrote the origin of species"])
assert len(cold_vectors) == 1 and len(cold_vectors[0]) == EMBEDDING_DIM

warm_latencies = []
for _ in range(3):
    _, warm_latency = timed_embed(["a warm follow-up request"])
    warm_latencies.append(warm_latency)

print(f"cold-start invocation: {cold_latency:.2f} s")
print(f"warm invocations: {', '.join(f'{value:.2f} s' for value in warm_latencies)}")
print(f"embedding dimensions: {len(cold_vectors[0])}")

### Visualize the cold-start overhead

The gap between the first (cold) call and the warm calls that follow is the
cold-start overhead: the time SageMaker spends starting a worker and loading
the model before it can serve a response. Plotting the latencies measured
above makes that gap concrete. After an idle stretch the endpoint scales back
to zero, so a later call pays the cold start again.

In [ ]:
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
import matplotlib.pyplot as plt

warm_mean = sum(warm_latencies) / len(warm_latencies)
overhead = cold_latency - warm_mean

labels = ["cold start\n(worker load + inference)", "warm\n(inference only)"]
values = [cold_latency, warm_mean]

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(labels, values, color=["#e07a5f", "#3d9a8b"])
ax.set_ylabel("latency (seconds)")
ax.set_title(f"Cold start adds ~{overhead:.2f}s of overhead")
ax.margins(y=0.15)
for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.2f}s",
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.show()

## Check the embeddings are usable

Latency is only half the story: the vectors also have to be good enough to rank
text by meaning. We embed a query together with a handful of candidate sentences
in one call, score each candidate against the query with cosine similarity, and
confirm the passage that actually answers the query comes out on top.


In [ ]:
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        raise ValueError("cosine similarity is undefined for a zero vector")
    return dot / (norm_a * norm_b)


assert cosine([1.0, 0.0], [0.0, 1.0]) == 0.0
assert abs(cosine([1.0, 2.0, 3.0], [1.0, 2.0, 3.0]) - 1.0) < 1e-9

query = "who wrote the origin of species"
candidates = [
    "Charles Darwin published On the Origin of Species in 1859, laying out his theory of evolution by natural selection.",
    "The Great Barrier Reef is the world's largest coral reef system, off the coast of Queensland, Australia.",
    "Python is a high-level programming language known for its readable syntax.",
    "Mount Kilimanjaro is the highest mountain in Africa.",
]

vectors = embed([query] + candidates)
query_vector, candidate_vectors = vectors[0], vectors[1:]

ranked = sorted(
    (
        {"score": cosine(query_vector, vector), "text": text}
        for text, vector in zip(candidates, candidate_vectors)
    ),
    key=lambda item: item["score"],
    reverse=True,
)

for hit in ranked:
    print(f"{hit['score']:.3f}  {hit['text']}")

assert ranked[0]["text"] == candidates[0]
print("\nretrieval smoke test passed: the Darwin passage ranks first")

## Clean up

Serverless endpoints cost nothing while idle, but the endpoint, its
configuration, and the model resource linger until you delete them, and a
provisioned-concurrency endpoint keeps billing for the reserved workers.
Deleting the endpoint also returns its share of your account's serverless
concurrency quota.


In [ ]:
def ignore_not_found(error):
    code = error.response.get("Error", {}).get("Code", "")
    message = error.response.get("Error", {}).get("Message", "")
    return code in {"ResourceNotFound", "ResourceNotFoundException"} or "not exist" in message


def cleanup_resources():
    print("deleting endpoint")
    try:
        sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
        sm.get_waiter("endpoint_deleted").wait(
            EndpointName=ENDPOINT_NAME,
            WaiterConfig={"Delay": 15, "MaxAttempts": 60},
        )
    except ClientError as error:
        if not ignore_not_found(error):
            raise

    print("deleting endpoint config")
    try:
        sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    except ClientError as error:
        if not ignore_not_found(error):
            raise

    print("deleting model")
    try:
        sm.delete_model(ModelName=model_name)
    except ClientError as error:
        if not ignore_not_found(error):
            raise


if CLEANUP:
    cleanup_resources()
else:
    print(f"left endpoint running: {ENDPOINT_NAME}")